In [ ]:
import random 
import string
import requests
import csv
import numpy as np
from datetime import datetime, UTC
import time
import pandas as pd

### Choose subset to scrap

In [ ]:
X = 2

Load usernames to scrap

In [ ]:
with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

df = pd.read_csv('data.csv', sep=',')

Scrap content 

In [ ]:
random_sleeps = np.random.uniform(low=0.35, high=0.65, size=100_000_000)

with open("data.csv", "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    for i, username in enumerate([usernames[0]]):
        j = 1
        headuser = "Mozilla/5.0 (compatible; scraper/1.0)"
        donnees_user = []
        username = username.strip('\n')
        after, to_cont = None, True

        while to_cont:
            headuser += random.choice(string.ascii_uppercase + string.digits)
            headers = {
                "User-Agent": headuser
            }
            session = requests.Session()
            session.headers.update(headers)
            params = params = {
                    "limit": 50
                }
            
            if after:
                params["after"] = after

            
            # r = session.get(
            #         f"https://www.reddit.com/user/{username}/about.json", 
            #         params=params,
            #         timeout=3)
            # date_ins = date_regis = datetime.utcfromtimestamp(r.json()["data"]["created_utc"])

            r = session.get(
                    f"https://www.reddit.com/user/{username}/.json",
                    params=params,
                    timeout=3)
            data = r.json()
            if data["data"]["after"]:
                after = data["data"]["after"]
            else : 
                to_cont = False
            print(after)
            if data['data']['children']:
                for child in data['data']['children']:
                    id = str(i+1).zfill(4) + str(j)
                    prod = child['data']
                    prod_created = datetime.fromtimestamp(prod['created_utc'], UTC)

                    if child['kind']=='t1': # comment 
                        body = str(prod['body'].replace("\n", "\\n"))
                        if prod['parent_id'].startswith('t1'): # comment to comment (= response)
                            type_prod = 'response'
                            r = session.get(
                                f"https://www.reddit.com/api/info.json?id={prod['parent_id']}",
                                timeout=3)
                            parent = r.json()['data']['children'][0]['data']
                            parent_user = parent['author']
                            parent_body = str(parent['body'].replace("\n", "\\n"))
                        elif prod['parent_id'].startswith('t3') : # Comment to post (=comment)
                            type_prod = 'comment'
                            parent_user = np.nan
                            parent_body = np.nan
                        
                        post_title = prod['link_title']
                        r = session.get(
                                f"https://www.reddit.com/api/info.json?id={prod['link_id']}",
                                timeout=3)
                        post_body = str(r.json()['data']['children'][0]['data']['selftext'].replace("\n", "\\n"))

                    elif child['kind']=='t3': # post
                        type_prod = 'post'
                        body = str(prod['selftext'].replace("\n", "\\n"))
                        parent_user = np.nan
                        parent_body = np.nan
                        post_title = prod['title']
                        post_body = 'self body'
                
                    interaction = [f'{X}X{id}', username, np.nan, type_prod, prod_created,
                        body, parent_user, parent_body, np.nan, # np.nan corresponds to registration date and parent's pp
                        post_title, post_body] 
                    donnees_user.append(interaction)

                    j += 1
                    if j%10 == 0:
                        time.sleep(random_sleeps[j])
                        print(j)
                    

        print(f'User {i+1} scaped with {j} prods.')
        writer.writerows(donnees_user)

In [ ]:
"""import langid

texte = "Ceci est un test en français."
langue, score = langid.classify(texte)

print(langue, score)  # fr, score de confiance"""